# Texas RRC Form R-1 Extraction Pipeline

This notebook provides an end-to-end pipeline for:
1. **Downloading** Form R-1 PDFs from the RRC website
2. **Extracting** structured data (auto-detecting digital vs scanned PDFs)
3. **Validating** using the material balance equation
4. **Comparing** against the existing analyst database (`output_cleaned_v2_1.xlsx`)

### What's in the existing database?
- **28,103 rows** across **15 facilities**, Jan 2014 – Dec 2025
- 24 product codes per facility-month (material balance from Page 1)
- **180 failed PDFs** across 27 facilities (mostly `FileDataError` from scanned/corrupt files)
- Key gap: Blanchard 03-0052 June 2025 is in the DB but shows **all zeros** — extraction failed silently
- ExxonMobil Beaumont (03-0138) is **entirely missing** from the database (only in failed list)

### Requirements
```bash
pip install pdfplumber pytesseract Pillow pandas requests beautifulsoup4 openpyxl PyMuPDF
# Windows: also install Tesseract OCR from https://github.com/UB-Mannheim/tesseract/wiki
# Then set: pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
```

**Note:** The previous version used `pdftoppm` (poppler) for rasterizing scanned PDFs, which doesn't work on Windows.
This version uses **PyMuPDF** (`fitz`) instead — pure Python, cross-platform, no system dependencies.

---
## 0. Setup & Configuration

In [1]:
import os
import re
import tempfile
from pathlib import Path
from datetime import datetime

import pandas as pd
import pdfplumber
import fitz  # PyMuPDF — replaces poppler/pdftoppm for rasterizing scanned PDFs
import pytesseract
from PIL import Image
import requests
from bs4 import BeautifulSoup

# ── Config ──
RAW_DATA_DIR = Path('Raw Data/Texas RRC')
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

EXISTING_DB_PATH = 'output_cleaned_v2.xlsx'
FAILED_DB_PATH = 'failed_data.xlsx'

# Windows Tesseract path — uncomment and adjust if needed:
pytesseract.pytesseract.tesseract_cmd = r'C:\Users\luiscarlos.gaitan\AppData\Local\Programs\Tesseract-OCR\tesseract.exe'

PRODUCT_CODES = {
    1: 'Propane', 2: 'Butane', 3: 'Butane-Propane', 4: 'Motor Gasoline',
    5: 'Kerosene', 6: 'Home Heating Oil', 7: 'Diesel Fuel',
    8: 'Other Middle Distillates', 9: 'Aviation Gasoline',
    10: 'Kerosene-Type Jet Fuel', 11: 'Naphtha-Type Jet Fuel',
    12: 'Fuel Oils #4 For Utility Use', 13: 'Fuel Oils #5,#6 For Utility Use',
    14: 'Fuel Oils #4 For Non-Utility Use', 15: 'Fuel Oils #5,#6 For Non-Utility Use',
    16: 'Bunker C', 17: 'Navy Special', 18: 'Other Residual Fuels',
    19: 'Petrochemical Feedstock', 20: 'Lubricants', 21: 'Special Naphthas',
    22: 'Solvent Products', 23: 'Miscellaneous', 24: 'Crude Oil',
}

print('Setup complete.')

Setup complete.


---
## 1. Download PDFs from RRC Website

The RRC publishes monthly refinery statements at predictable URLs.
This cell scrapes the index page for each month, finds PDF links, and downloads them.

**Note:** The URL format changed in 2020 (month padding). The code handles both.

In [2]:
def build_rrc_url(year: int, month: int) -> str:
    """Build the RRC index page URL for a given year/month."""
    if year < 2020:
        return (f'https://www.rrc.texas.gov/oil-and-gas/research-and-statistics/'
                f'refinery-statements/refineries-statements-{year}/'
                f'refinery-statements-{month:02d}-{year}')
    else:
        return (f'https://www.rrc.texas.gov/oil-and-gas/research-and-statistics/'
                f'refinery-statements/refineries-statements-{year}/'
                f'refinery-statements-{month}-{year}')


def scrape_pdf_links(year: int, month: int) -> pd.DataFrame:
    """Scrape the RRC page and return a DataFrame of facility IDs + PDF links."""
    url = build_rrc_url(year, month)
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.text, 'html.parser')
    table = soup.find('table', {'class': 'table table-bordered'})
    if table is None:
        print(f"  WARNING: No table found at {url}")
        return pd.DataFrame()
    
    rows = []
    for tr in table.find_all('tr')[1:]:  # skip header
        tds = tr.find_all('td')
        if len(tds) < 2:
            continue
        facility_id = tds[0].text.strip()
        operator = tds[1].text.strip() if len(tds) > 1 else ''
        
        # Find the PDF link (could be on either td)
        link = None
        for td in tds:
            a = td.find('a')
            if a and a.get('href'):
                link = a['href']
                if not link.startswith('http'):
                    link = 'https://www.rrc.texas.gov' + link
                break
        
        if link and facility_id:
            rows.append({
                'facility_id': facility_id,
                'operator': operator,
                'pdf_url': link,
                'year': year,
                'month': month,
            })
    
    return pd.DataFrame(rows)


def download_pdfs(year_start: int, year_end: int, month_start: int = 1, month_end: int = 12,
                  facility_filter: list = None) -> pd.DataFrame:
    """
    Download R-1 PDFs for a date range.
    
    Args:
        year_start, year_end: Inclusive year range
        month_start, month_end: Month range (default: full year)
        facility_filter: Optional list of facility IDs to download (e.g. ['03-0052', '03-0138'])
    
    Returns:
        DataFrame log of all downloads with status
    """
    log = []
    
    for year in range(year_start, year_end + 1):
        ms = month_start if year == year_start else 1
        me = month_end if year == year_end else 12
        
        for month in range(ms, me + 1):
            print(f"Scraping {year}-{month:02d}...", end=" ")
            try:
                links = scrape_pdf_links(year, month)
            except Exception as e:
                print(f"FAILED: {e}")
                log.append({'year': year, 'month': month, 'status': 'scrape_failed', 'error': str(e)})
                continue
            
            if links.empty:
                print("no data")
                continue
            
            if facility_filter:
                links = links[links['facility_id'].isin(facility_filter)]
            
            print(f"{len(links)} PDFs")
            
            for _, row in links.iterrows():
                fid = row['facility_id']
                op = row['operator']
                url = row['pdf_url']
                
                out_dir = RAW_DATA_DIR / fid
                out_dir.mkdir(parents=True, exist_ok=True)
                fname = f"{year}_{month}___{fid}_{op}.pdf"
                out_path = out_dir / fname
                
                if out_path.exists():
                    log.append({**row, 'local_path': str(out_path), 'status': 'already_exists'})
                    continue
                
                try:
                    resp = requests.get(url, timeout=60)
                    resp.raise_for_status()
                    out_path.write_bytes(resp.content)
                    log.append({**row, 'local_path': str(out_path), 'status': 'downloaded'})
                except Exception as e:
                    log.append({**row, 'local_path': str(out_path), 'status': 'download_failed', 'error': str(e)})
    
    return pd.DataFrame(log)

In [3]:
# ── EXAMPLE: Download a specific range ──
# Uncomment and adjust as needed:

# Download everything for 2024-2025:
# download_log = download_pdfs(2024, 2025)

# Download only specific facilities:
# download_log = download_pdfs(2024, 2025, facility_filter=['03-0052', '03-0138', '03-0012'])

# Download a single month:
# download_log = download_pdfs(2025, 2025, month_start=6, month_end=6)

# download_log.to_csv(OUTPUT_DIR / 'download_log.csv', index=False)
# print(download_log['status'].value_counts())

print("Download cell ready. Uncomment the lines above to run.")

Download cell ready. Uncomment the lines above to run.


---
## 2. PDF Extraction Engine

The extractor auto-detects whether a PDF is digital (text-based) or scanned (image-based) and uses the appropriate method:

- **Digital PDFs** → `pdfplumber` table extraction (fast, ~100% accurate)
- **Scanned PDFs** → `pytesseract` OCR with positional column mapping (~85% accurate, needs validation)

The material balance equation is used as a built-in validation check:
> `Storage_BOM + Receipts + Products_Mfg = Input_Run_Blends + Fuel_Used + Deliveries + Storage_EOM`

In [4]:
# ── Helpers ──

def parse_number(raw) -> float:
    """Clean a number from OCR or pdfplumber. Returns 0.0 for blanks/dashes."""
    if raw is None:
        return 0.0
    s = str(raw).strip()
    if not s or s in ('-', '=', ':', '', 'E', 'R', 'a', 's', '>', '\u00ae', 'Fz'):
        return 0.0
    s = s.replace(',', '').replace(' ', '')
    s = re.sub(r'[^0-9.\-]', '', s)
    if not s or s == '-':
        return 0.0
    try:
        return float(s)
    except ValueError:
        return 0.0


def is_digital_pdf(pdf_path: str) -> bool:
    """Returns True if the PDF has extractable text (not a scanned image)."""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = pdf.pages[0].extract_text() or ''
            return len(text.strip()) > 200
    except Exception:
        return False


def extract_header_from_text(text: str) -> dict:
    """Parse operator, month, registration number from page text."""
    info = {}
    m = re.search(r'Operator[:\s]+(.+?)(?:Month|$)', text, re.IGNORECASE)
    if m: info['operator'] = m.group(1).strip().rstrip('.')
    
    m = re.search(r'(January|February|March|April|May|June|July|August|'
                   r'September|October|November|December)\s+\d{4}', text, re.IGNORECASE)
    if m: info['month_str'] = m.group(0)
    
    m = re.search(r'(\d{2}-\d{4})', text)
    if m: info['registration_no'] = m.group(1)
    
    m = re.search(r'Refinery\s*(?:Name)?\s*[:.]?\s*(.+?)(?:\n|$)', text, re.IGNORECASE)
    if m: info['refinery_name'] = m.group(1).strip()
    
    return info

In [5]:
# ── Digital PDF Extraction ──

def extract_digital_pdf(pdf_path: str) -> dict:
    """Extract Page 1 material balance from a digital (text-based) PDF."""
    with pdfplumber.open(pdf_path) as pdf:
        page1_text = pdf.pages[0].extract_text() or ''
        header = extract_header_from_text(page1_text)
        
        tables = pdf.pages[0].extract_tables()
        if not tables:
            return {'header': header, 'data': [], 'mode': 'digital', 'error': 'No table found on page 1'}
        
        t = tables[0]
        rows = []
        for row in t[1:]:  # skip header row
            if row[0] and 'TOTALS' in str(row[0]):
                continue
            code_val = parse_number(row[1])
            if code_val == 0:
                continue
            code = int(code_val)
            rows.append({
                'Code': code,
                'Storage Beginning of Month': parse_number(row[2]),
                'Receipts': parse_number(row[3]),
                'Input': parse_number(row[4]),
                'Products Manufactured': parse_number(row[5]),
                'Fuel Used': parse_number(row[6]),
                'Deliveries': parse_number(row[7]),
                'Storage End of Month': parse_number(row[8]),
            })
        
        return {'header': header, 'data': rows, 'mode': 'digital', 'error': None}

In [6]:
# ── Scanned PDF Extraction (OCR via PyMuPDF + Tesseract) ──

# Column boundaries for Form R-1 at 200 DPI
OCR_Y_START, OCR_Y_END, OCR_ROW_H = 330, 1026, 28
OCR_COLS = [
    ('material_name',    0,    520),
    ('code',             520,  610),
    ('storage_bom',      610,  790),
    ('receipts',         790,  1000),
    ('input',            1000, 1220),
    ('products_mfg',     1220, 1470),
    ('fuel_used',        1470, 1700),
    ('deliveries',       1700, 1920),
    ('storage_eom',      1920, 2200),
]

def rasterize_page(pdf_path: str, page_num: int = 0, dpi: int = 200) -> Image.Image:
    """
    Rasterize a single PDF page to a PIL Image using PyMuPDF.
    Works on Windows/Mac/Linux — no poppler/pdftoppm needed.
    """
    doc = fitz.open(pdf_path)
    page = doc[page_num]
    # fitz uses a zoom matrix: 200 DPI = 200/72 zoom factor
    zoom = dpi / 72
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat)
    img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
    doc.close()
    return img


def extract_scanned_pdf(pdf_path: str) -> dict:
    """Extract Page 1 material balance from a scanned (image) PDF using OCR."""
    try:
        img = rasterize_page(pdf_path, page_num=0, dpi=200)
    except Exception as e:
        return {'header': {}, 'data': [], 'mode': 'scanned_ocr', 'error': f'Rasterize failed: {e}'}
    
    # Header
    header_text = pytesseract.image_to_string(img.crop((0, 0, img.width, 250)))
    header = extract_header_from_text(header_text)
    
    # Table via positional OCR
    data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DATAFRAME)
    data = data[data['text'].notna() & (data['text'].str.strip() != '')]
    table = data[(data['top'] >= OCR_Y_START) & (data['top'] <= OCR_Y_END)].copy()
    
    table['cx'] = table['left'] + table['width'] / 2
    
    def assign_col(cx):
        for name, lo, hi in OCR_COLS:
            if lo <= cx < hi:
                return name
        return 'unknown'
    
    table['col'] = table['cx'].apply(assign_col)
    table['row_bin'] = ((table['top'] - OCR_Y_START) // OCR_ROW_H).astype(int)
    
    # Build rows — use fixed row position to assign product codes
    rows = []
    for row_bin in sorted(table['row_bin'].unique()):
        if row_bin >= 24:
            break
        group = table[table['row_bin'] == row_bin]
        code = row_bin + 1
        
        def get_val(col_name):
            words = group[group['col'] == col_name].sort_values('left')
            return ' '.join(words['text'].astype(str).tolist())
        
        rows.append({
            'Code': code,
            'Storage Beginning of Month': parse_number(get_val('storage_bom')),
            'Receipts': parse_number(get_val('receipts')),
            'Input': parse_number(get_val('input')),
            'Products Manufactured': parse_number(get_val('products_mfg')),
            'Fuel Used': parse_number(get_val('fuel_used')),
            'Deliveries': parse_number(get_val('deliveries')),
            'Storage End of Month': parse_number(get_val('storage_eom')),
        })
    
    # Fill in any missing codes
    existing_codes = {r['Code'] for r in rows}
    for code in range(1, 25):
        if code not in existing_codes:
            rows.append({
                'Code': code,
                'Storage Beginning of Month': 0.0, 'Receipts': 0.0, 'Input': 0.0,
                'Products Manufactured': 0.0, 'Fuel Used': 0.0,
                'Deliveries': 0.0, 'Storage End of Month': 0.0,
            })
    rows.sort(key=lambda r: r['Code'])
    
    return {'header': header, 'data': rows, 'mode': 'scanned_ocr', 'error': None}

In [7]:
# ── Unified Extractor ──

def extract_r1_pdf(pdf_path: str) -> dict:
    """
    Extract Form R-1 data from any PDF (auto-detects digital vs scanned).
    Returns dict with 'header', 'data' (list of row dicts), 'mode', 'error'.
    """
    try:
        if is_digital_pdf(pdf_path):
            return extract_digital_pdf(pdf_path)
        else:
            return extract_scanned_pdf(pdf_path)
    except Exception as e:
        return {'header': {}, 'data': [], 'mode': 'error', 'error': str(e)}


def validate_balance(df: pd.DataFrame) -> pd.DataFrame:
    """
    Check: Storage_BOM + Receipts + Products_Mfg = Input + Fuel_Used + Deliveries + Storage_EOM
    """
    df = df.copy()
    inflows = df['Storage Beginning of Month'] + df['Receipts'] + df['Products Manufactured']
    outflows = df['Input'] + df['Fuel Used'] + df['Deliveries'] + df['Storage End of Month']
    df['balance_diff'] = inflows - outflows
    df['balanced'] = df['balance_diff'].abs() < 1
    return df

print("Extraction engine ready.")

Extraction engine ready.


---
## 3. Batch Extraction

Process all downloaded PDFs in the `Raw Data/Texas RRC/` directory.

In [8]:
def batch_extract(data_dir: Path = RAW_DATA_DIR, 
                  facility_filter: list = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Extract all PDFs in the data directory.
    
    Returns:
        (success_df, failed_df) — structured results and error log
    """
    all_rows = []
    failed = []
    
    pdf_files = sorted(data_dir.rglob('*.pdf'))
    if facility_filter:
        pdf_files = [p for p in pdf_files if any(f in str(p) for f in facility_filter)]
    
    print(f"Processing {len(pdf_files)} PDFs...")
    
    for i, pdf_path in enumerate(pdf_files):
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(pdf_files)}...")
        
        # Parse facility_id and date from filename: YYYY_M___ID_OPERATOR.pdf
        fname = pdf_path.stem
        parts = fname.split('___')
        if len(parts) < 2:
            failed.append({'file': str(pdf_path), 'error': 'Cannot parse filename'})
            continue
        
        date_part = parts[0]  # e.g. "2024_6"
        id_part = parts[1].split('_')[0]  # e.g. "03-0052"
        
        try:
            year, month = date_part.split('_')[:2]
            year, month = int(year), int(month)
            # Create end-of-month date to match existing DB format
            if month == 12:
                date = datetime(year, 12, 31)
            else:
                date = datetime(year, month + 1, 1) - pd.Timedelta(days=1)
        except (ValueError, IndexError):
            failed.append({'file': str(pdf_path), 'error': f'Cannot parse date from {date_part}'})
            continue
        
        result = extract_r1_pdf(str(pdf_path))
        
        if result['error'] or not result['data']:
            failed.append({
                'id': id_part, 'date': date, 'file': str(pdf_path),
                'mode': result['mode'], 'error': result.get('error', 'No data extracted'),
            })
            continue
        
        for row in result['data']:
            row['id'] = id_part
            row['date'] = date
            row['source_file'] = str(pdf_path)
            row['extraction_mode'] = result['mode']
            all_rows.append(row)
    
    success_df = pd.DataFrame(all_rows)
    failed_df = pd.DataFrame(failed)
    
    if not success_df.empty:
        success_df = validate_balance(success_df)
    
    print(f"\nDone. {len(success_df)} rows extracted, {len(failed_df)} failures.")
    return success_df, failed_df

In [9]:
# ── Run batch extraction ──
# Uncomment to run on your downloaded PDFs:

new_data, new_failures = batch_extract()

# Or extract only specific facilities:
# new_data, new_failures = batch_extract(facility_filter=['03-0052', '03-0138'])

# Save results:
new_data.to_csv(OUTPUT_DIR / 'extracted_data.csv', index=False)
new_failures.to_csv(OUTPUT_DIR / 'extraction_failures.csv', index=False)

print("Batch extraction cell ready. Uncomment to run.")

Processing 3707 PDFs...
  50/3707...
  100/3707...
  150/3707...
  200/3707...
  250/3707...
  300/3707...
  350/3707...
  400/3707...
  450/3707...
  500/3707...
  550/3707...
  600/3707...
  650/3707...
  700/3707...
  750/3707...
  800/3707...
  850/3707...
  900/3707...
  950/3707...
  1000/3707...
  1050/3707...
  1100/3707...
  1150/3707...
  1200/3707...
  1250/3707...
  1300/3707...
  1350/3707...
  1400/3707...
  1450/3707...
  1500/3707...
  1550/3707...
  1600/3707...
  1650/3707...
  1700/3707...
  1750/3707...
  1800/3707...
  1850/3707...
  1900/3707...
  1950/3707...
  2000/3707...
  2050/3707...
  2100/3707...
  2150/3707...
  2200/3707...
  2250/3707...
  2300/3707...
  2350/3707...
  2400/3707...
  2450/3707...
  2500/3707...
  2550/3707...
  2600/3707...
  2650/3707...
  2700/3707...
  2750/3707...
  2800/3707...
  2850/3707...
  2900/3707...
  2950/3707...
  3000/3707...
  3050/3707...
  3100/3707...
  3150/3707...
  3200/3707...
  3250/3707...
  3300/3707...
  3350

In [35]:
new_data[new_data['balance_diff'] > 100]['balanced'].value_counts()/len(new_data)

balanced
False    0.080437
Name: count, dtype: float64

In [18]:
new_data['balanced'].value_counts()/len(new_data)

balanced
True     0.771532
False    0.228468
Name: count, dtype: float64

---
## 4. Compare Against Existing Database

This section loads the existing analyst database and compares your fresh extraction
against it, flagging:
- **Mismatches**: where the numbers differ (possible OCR errors or data corrections)
- **Silent zeros**: rows in the existing DB that are all-zero but shouldn't be
- **Missing data**: facility-months that exist in one source but not the other
- **New coverage**: data you've captured that wasn't in the original database

In [10]:
def load_existing_db(path: str = EXISTING_DB_PATH) -> pd.DataFrame:
    """Load and prepare the existing analyst database."""
    df = pd.read_excel(path)
    df['date'] = pd.to_datetime(df['date'])
    return df


def compare_extractions(new_df: pd.DataFrame, existing_df: pd.DataFrame,
                        tolerance: float = 1.0) -> dict:
    """
    Compare new extraction against existing database.
    
    Returns dict with:
        'matches': rows that agree within tolerance
        'mismatches': rows that differ
        'new_only': rows only in new extraction
        'existing_only': rows only in existing DB
        'silent_zeros': existing rows that are all-zero but new extraction has data
    """
    value_cols = ['Storage Beginning of Month', 'Receipts', 'Input',
                  'Products Manufactured', 'Fuel Used', 'Deliveries', 'Storage End of Month']
    
    merge_keys = ['id', 'date', 'Code']
    
    merged = new_df.merge(existing_df, on=merge_keys, how='outer',
                          suffixes=('_new', '_old'), indicator=True)
    
    # Categorize
    new_only = merged[merged['_merge'] == 'left_only'].copy()
    existing_only = merged[merged['_merge'] == 'right_only'].copy()
    both = merged[merged['_merge'] == 'both'].copy()
    
    # Check for mismatches
    matches = []
    mismatches = []
    silent_zeros = []
    
    for _, row in both.iterrows():
        is_match = True
        old_all_zero = True
        new_has_data = False
        diffs = {}
        
        for col in value_cols:
            new_val = row.get(f'{col}_new', 0) or 0
            old_val = row.get(f'{col}_old', 0) or 0
            
            if old_val != 0:
                old_all_zero = False
            if new_val != 0:
                new_has_data = True
            
            if abs(new_val - old_val) > tolerance:
                is_match = False
                diffs[col] = {'new': new_val, 'old': old_val, 'diff': new_val - old_val}
        
        if old_all_zero and new_has_data:
            silent_zeros.append({**row.to_dict(), 'diffs': diffs})
        elif is_match:
            matches.append(row.to_dict())
        else:
            mismatches.append({**row.to_dict(), 'diffs': diffs})
    
    return {
        'matches': pd.DataFrame(matches),
        'mismatches': pd.DataFrame(mismatches) if mismatches else pd.DataFrame(),
        'new_only': new_only,
        'existing_only': existing_only,
        'silent_zeros': pd.DataFrame(silent_zeros) if silent_zeros else pd.DataFrame(),
    }


def print_comparison_summary(comparison: dict):
    """Print a readable summary of the comparison results."""
    print("=" * 60)
    print("COMPARISON SUMMARY")
    print("=" * 60)
    print(f"  Matching rows:      {len(comparison['matches'])}")
    print(f"  Mismatches:         {len(comparison['mismatches'])}")
    print(f"  Silent zeros fixed: {len(comparison['silent_zeros'])}")
    print(f"  New data only:      {len(comparison['new_only'])}")
    print(f"  Existing DB only:   {len(comparison['existing_only'])}")
    
    if not comparison['mismatches'].empty:
        print(f"\n--- Top Mismatches ---")
        mm = comparison['mismatches']
        for _, row in mm.head(20).iterrows():
            print(f"  {row['id']} | {row['date']} | Code {row['Code']} "
                  f"({PRODUCT_CODES.get(int(row['Code']), '?')})")
            if 'diffs' in row and isinstance(row['diffs'], dict):
                for col, d in row['diffs'].items():
                    print(f"    {col}: old={d['old']:,.0f}  new={d['new']:,.0f}  diff={d['diff']:,.0f}")
    
    if not comparison['silent_zeros'].empty:
        print(f"\n--- Silent Zeros (existing DB has zeros, we found real data) ---")
        sz = comparison['silent_zeros']
        # Group by facility-month
        groups = sz.groupby(['id', 'date']).size()
        for (fid, dt), count in groups.items():
            print(f"  {fid} | {dt} | {count} codes with data")

print("Comparison tools ready.")

Comparison tools ready.


In [11]:
# ── Run comparison ──
# After running batch extraction above, uncomment:

existing_db = load_existing_db()
comparison = compare_extractions(new_data, existing_db)
print_comparison_summary(comparison)

# Save detailed comparison:
comparison['mismatches'].to_csv(OUTPUT_DIR / 'mismatches.csv', index=False)
comparison['silent_zeros'].to_csv(OUTPUT_DIR / 'silent_zeros.csv', index=False)
comparison['new_only'].to_csv(OUTPUT_DIR / 'new_data.csv', index=False)

print("Comparison cell ready. Uncomment after running extraction.")

COMPARISON SUMMARY
  Matching rows:      15045
  Mismatches:         6341
  Silent zeros fixed: 4371
  New data only:      52422
  Existing DB only:   2586

--- Top Mismatches ---
  01-0692 | 2020-01-31 00:00:00 | Code 10 (Kerosene-Type Jet Fuel)
    Products Manufactured: old=0  new=30,493  diff=30,493
    Deliveries: old=0  new=23,439  diff=23,439
  01-0692 | 2020-01-31 00:00:00 | Code 14 (Fuel Oils #4 For Non-Utility Use)
    Products Manufactured: old=0  new=18,590  diff=18,590
    Deliveries: old=0  new=17,045  diff=17,045
  01-0692 | 2020-01-31 00:00:00 | Code 22 (Solvent Products)
    Products Manufactured: old=-1  new=1  diff=2
  01-0692 | 2020-02-29 00:00:00 | Code 10 (Kerosene-Type Jet Fuel)
    Products Manufactured: old=0  new=33,867  diff=33,867
    Deliveries: old=0  new=36,018  diff=36,018
  01-0692 | 2020-04-30 00:00:00 | Code 7 (Diesel Fuel)
    Storage Beginning of Month: old=39,846  new=0  diff=-39,846
    Products Manufactured: old=187,729  new=0  diff=-187,729
    

---
## 5. Quick Test: Extract the Two Sample PDFs

Test the pipeline on the two PDFs we've been examining.

In [12]:
# ── Test on the Blanchard (digital) PDF ──
# Update path to wherever your PDF is:
blanchard_path = '2025-june-03-0052.pdf'  # Adjust as needed

if os.path.exists(blanchard_path):
    result = extract_r1_pdf(blanchard_path)
    print(f"Mode: {result['mode']}")
    print(f"Header: {result['header']}")
    
    df = pd.DataFrame(result['data'])
    df = validate_balance(df)
    df['material'] = df['Code'].map(PRODUCT_CODES)
    
    print(f"\nBalance check: {df['balanced'].sum()}/24 pass")
    print(df[['Code', 'material', 'Storage Beginning of Month', 'Receipts',
              'Input', 'Products Manufactured', 'Fuel Used', 'Deliveries',
              'Storage End of Month', 'balanced']].to_string(index=False))
    
    # Compare to existing DB
    existing_db = load_existing_db()
    existing_blanchard = existing_db[
        (existing_db['id'] == '03-0052') & 
        (existing_db['date'] == '2025-06-30')
    ]
    
    if not existing_blanchard.empty:
        print(f"\n--- Existing DB has {len(existing_blanchard)} rows for this filing ---")
        has_data = (existing_blanchard[['Storage Beginning of Month', 'Receipts',
                                        'Products Manufactured', 'Deliveries']].sum().sum())
        if has_data == 0:
            print("WARNING: Existing DB shows ALL ZEROS — previous extraction failed silently!")
            print("Your new extraction recovered the real data.")
else:
    print(f"File not found: {blanchard_path}")
    print("Update the path above to point to your PDF.")

Mode: digital
Header: {'operator': 'Blanchard Refining', 'month_str': 'June 2025', 'registration_no': '03-0052', 'refinery_name': 'Galveston Bay Refinery'}

Balance check: 23/24 pass
 Code                            material  Storage Beginning of Month   Receipts      Input  Products Manufactured  Fuel Used  Deliveries  Storage End of Month  balanced
    1                             Propane                       551.0     3037.0        0.0               358477.0        0.0    361519.0                 546.0      True
    2                              Butane                     54441.0   661667.0   676445.0               517577.0        0.0    519512.0               37729.0     False
    3                      Butane-Propane                         0.0        0.0        0.0                    0.0        0.0         0.0                   0.0      True
    4                      Motor Gasoline                   2330517.0     4043.0    94504.0              9037415.0        0.0   8917103.0

In [13]:
# ── Test on the ExxonMobil Beaumont (scanned) PDF ──
exxon_path = '2026-january-03-0138.pdf'  # Adjust as needed

if os.path.exists(exxon_path):
    result = extract_r1_pdf(exxon_path)
    print(f"Mode: {result['mode']}")
    print(f"Header: {result['header']}")
    
    if result['data']:
        df = pd.DataFrame(result['data'])
        df = validate_balance(df)
        df['material'] = df['Code'].map(PRODUCT_CODES)
        
        n_active = len(df[df[['Storage Beginning of Month', 'Receipts',
                              'Products Manufactured', 'Deliveries']].sum(axis=1) > 0])
        n_balanced = df.loc[df[['Storage Beginning of Month', 'Receipts',
                                'Products Manufactured', 'Deliveries']].sum(axis=1) > 0, 'balanced'].sum()
        
        print(f"\nBalance check: {n_balanced}/{n_active} active rows pass")
        print("\nRows that FAILED balance check (likely OCR errors):")
        active = df[df[['Storage Beginning of Month', 'Receipts',
                        'Products Manufactured', 'Deliveries']].sum(axis=1) > 0]
        fails = active[~active['balanced']]
        if not fails.empty:
            print(fails[['Code', 'material', 'balance_diff']].to_string(index=False))
            print("\nThese rows need manual review or cell-level OCR refinement.")
    else:
        print(f"Extraction failed: {result['error']}")
else:
    print(f"File not found: {exxon_path}")
    print("Update the path above to point to your PDF.")

Mode: scanned_ocr
Header: {'month_str': 'January 2026', 'registration_no': '03-0138', 'refinery_name': 'Form R-1'}

Balance check: 3/11 active rows pass

Rows that FAILED balance check (likely OCR errors):
 Code               material  balance_diff
    1                Propane  1.000000e+00
    2                 Butane  5.933900e+04
    4         Motor Gasoline -1.000000e+00
    7            Diesel Fuel  1.000000e+00
   10 Kerosene-Type Jet Fuel  1.000000e+00
   20             Lubricants  3.999900e+04
   23          Miscellaneous  8.526982e+06
   24              Crude Oil -1.735418e+07

These rows need manual review or cell-level OCR refinement.


In [14]:
df = pd.DataFrame(result['data'])
df = validate_balance(df)
df['material'] = df['Code'].map(PRODUCT_CODES)
df['id'] = '03-0138'
df['date'] = pd.Timestamp('2026-01-31')

# Flag confidence level
df['confidence'] = 'high'
df.loc[~df['balanced'], 'confidence'] = 'needs_review'

# Display
df[['Code', 'material', 'Storage Beginning of Month', 'Receipts', 'Input',
    'Products Manufactured', 'Fuel Used', 'Deliveries', 'Storage End of Month',
    'balance_diff', 'confidence']]

,Code,material,Storage Beginning of Month,Receipts,Input,Products Manufactured,Fuel Used,Deliveries,Storage End of Month,balance_diff,confidence
0,1,Propane,1787.0,1.026640e+05,102991.0,1212370.0,0.0,1212370.0,1459.0,1.000000e+00,needs_review
1,2,Butane,77129.0,7.630800e+04,17337.0,736108.0,0.0,735853.0,77016.0,5.933900e+04,needs_review
2,3,Butane-Propane,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.000000e+00,high
3,4,Motor Gasoline,1873591.0,3.059530e+05,437714.0,7449382.0,0.0,7434665.0,1756548.0,-1.000000e+00,needs_review
4,5,Kerosene,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.000000e+00,high
5,6,Home Heating Oil,0.0,0.000000e+00,0.0,290110.0,0.0,290110.0,0.0,0.000000e+00,high
6,7,Diesel Fuel,1187415.0,2.284000e+03,2284.0,5549241.0,1293.0,5907376.0,827986.0,1.000000e+00,needs_review
7,8,Other Middle Distillates,0.0,4.795000e+03,4795.0,0.0,0.0,0.0,0.0,0.000000e+00,high
8,9,Aviation Gasoline,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.000000e+00,high
9,10,Kerosene-Type Jet Fuel,462023.0,0.000000e+00,0.0,2240962.0,0.0,2214231.0,488753.0,1.000000e+00,needs_review


---
## 6. Build Final Database

Merge your new extraction with the existing database, keeping the best value for each cell.

In [15]:
def build_merged_database(new_df: pd.DataFrame, existing_df: pd.DataFrame) -> pd.DataFrame:
    """
    Merge new extraction with existing DB.
    Priority: new extraction wins if existing is all-zeros, otherwise keep existing
    (since it was presumably validated by the previous analyst).
    New-only rows are appended.
    """
    value_cols = ['Storage Beginning of Month', 'Receipts', 'Input',
                  'Products Manufactured', 'Fuel Used', 'Deliveries', 'Storage End of Month']
    merge_keys = ['id', 'date', 'Code']
    
    merged = existing_df.merge(new_df[merge_keys + value_cols],
                               on=merge_keys, how='outer',
                               suffixes=('', '_new'))
    
    # For rows where existing is all-zero but new has data, use new values
    for col in value_cols:
        new_col = f'{col}_new'
        if new_col in merged.columns:
            # Replace zeros with new data
            old_zero_mask = merged[col].fillna(0) == 0
            new_has_data = merged[new_col].fillna(0) != 0
            merged.loc[old_zero_mask & new_has_data, col] = merged.loc[
                old_zero_mask & new_has_data, new_col]
            
            # Fill NaN (new-only rows) with new values
            merged[col] = merged[col].fillna(merged[new_col])
            merged.drop(columns=[new_col], inplace=True)
    
    # Clean up
    merged = merged.sort_values(merge_keys).reset_index(drop=True)
    
    return merged


# ── Build and save ──
# Uncomment after running extraction:

# existing_db = load_existing_db()
# final_db = build_merged_database(new_data, existing_db)
# final_db.to_excel(OUTPUT_DIR / 'rrc_r1_database_merged.xlsx', index=False)
# final_db.to_csv(OUTPUT_DIR / 'rrc_r1_database_merged.csv', index=False)

# print(f"Final database: {len(final_db)} rows")
# print(f"Facilities: {final_db['id'].nunique()}")
# print(f"Date range: {final_db['date'].min()} to {final_db['date'].max()}")

print("Database merge cell ready. Uncomment after extraction.")

Database merge cell ready. Uncomment after extraction.


---
## 7. Known Issues & Next Steps

### Current limitations

| Issue | Impact | Workaround |
|-------|--------|------------|
| Scanned PDFs have ~85% OCR accuracy | Misread digits on some rows | Balance check flags bad rows; manual review or cell-crop OCR |
| Existing DB has silent zero-fills | Missing data looks like real data | Comparison tool identifies these |
| Receipts/Deliveries pages not parsed | Counterparty detail not in DB | Can add pdfplumber text extraction for digital PDFs |
| Capacity data not extracted for scanned | Can't compute utilization for all facilities | Add OCR for page 4 |
| ExxonMobil (03-0138) entirely missing | Major USGC refinery not in DB | Scanned PDF — needs OCR or manual entry |

### Recommended next steps

1. **Run the download + extract pipeline** on the full 2014–2025 range
2. **Review the balance check failures** — these are your OCR error candidates
3. **Cross-check against EIA weekly data** for aggregate PADD 3 totals as a sanity check
4. **Add counterparty parsing** from Pages 2–3 (digital PDFs are straightforward)
5. **Build Dash/Streamlit dashboard** on top of the merged database
6. **Schedule monthly updates** — new filings appear ~6 weeks after month-end